# 0 · Fundamentos: agente, entorno y la API de Gymnasium

*Curso: "Python Reinforcement Learning using Gymnasium – Full Course" (freeCodeCamp, Mustafa Esoofally).
Secciones 0:00 – 0:33 del video.*

## La idea en una frase
Un **agente** observa el **estado** de un **entorno**, toma una **acción**, y el entorno le
devuelve una **recompensa** y un nuevo estado. El objetivo es aprender una **política**
(una regla estado → acción) que maximice la recompensa **acumulada**, no la inmediata.

```
       ┌──────────── acción a_t ────────────┐
       │                                    ▼
  ┌─────────┐                         ┌───────────┐
  │ AGENTE  │                         │  ENTORNO  │
  │ política│◄── estado s_{t+1} ──────│           │
  └─────────┘◄── recompensa r_{t+1} ──└───────────┘
```

## Puente con el marco de Powell (video 1 del módulo)
Powell describe *cualquier* problema de decisión secuencial con 5 elementos. Gymnasium los
implementa casi uno a uno:

| Powell | Gymnasium |
|---|---|
| Estado $S_t$ | `observation` |
| Decisión $x_t$ | `action` |
| Información exógena $W_{t+1}$ | lo aleatorio dentro de `env.step` (la carta que sale, el ruido físico) |
| Función de transición $S_{t+1} = S^M(S_t, x_t, W_{t+1})$ | `env.step(action)` |
| Función objetivo $\max_\pi \mathbb{E}\sum_t C(S_t, X^\pi(S_t))$ | maximizar la suma de `reward` |

Y la **política** $X^\pi(S_t)$ es lo que nosotros programamos. En este curso usamos
dos políticas de la clase **VFA** (*Value Function Approximation*) de Powell:
- Blackjack: una tabla de valores $Q(s,a)$ → política $\varepsilon$-greedy sobre $Q$.
- CartPole: una red neuronal que aproxima $Q(s,a)$ → DQN.

In [1]:
import gymnasium as gym
import numpy as np

## Crear un entorno y mirar sus "espacios"
Cada entorno declara qué observaciones produce (`observation_space`) y qué acciones acepta
(`action_space`). `Discrete(n)` = enteros 0..n-1; `Box` = vector de números reales con límites.

- CartPole: observación = posición, velocidad, ángulo, velocidad angular. Acción 0 = izquierda, 1 = derecha.
- Blackjack: observación = (suma del jugador, carta del dealer, as usable). Acción 0 = plantarse, 1 = pedir.

In [2]:
env = gym.make("CartPole-v1")
print(env.observation_space)
print(env.action_space)

env_bj = gym.make("Blackjack-v1")
print(env_bj.observation_space)
print(env_bj.action_space)

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Discrete(2)
Tuple(Discrete(32), Discrete(11), Discrete(2))
Discrete(2)


## El ciclo básico: `reset` → `step` → … → fin
`env.step(a)` regresa 5 cosas:
- `observation`: el nuevo estado
- `reward`: la recompensa de ese paso
- `terminated`: el episodio acabó **por la dinámica** (el palo se cayó, la mano terminó)
- `truncated`: se cortó **por tiempo** (CartPole corta a los 500 pasos)
- `info`: diccionario con datos extra de diagnóstico

La diferencia terminated/truncated importa: si fue *truncated*, el estado siguiente aún
tenía valor futuro; si fue *terminated*, no. Lo usaremos en Q-learning y DQN.

In [3]:
obs, info = env.reset(seed=42)
total = 0
done = False

while not done:
    action = env.action_space.sample()  # acción al azar
    obs, reward, terminated, truncated, info = env.step(action)
    total += reward
    done = terminated or truncated

print("Recompensa total:", total)

Recompensa total: 12.0


## ¿Qué tan malo es actuar al azar?
Línea base para comparar después: corremos 200 episodios con acciones aleatorias.

In [4]:
def jugar(politica, n=200):
    puntajes = []
    for i in range(n):
        obs, info = env.reset(seed=i)
        total = 0
        done = False
        while not done:
            obs, reward, terminated, truncated, info = env.step(politica(obs))
            total += reward
            done = terminated or truncated
        puntajes.append(total)
    return np.mean(puntajes)

print("Aleatoria:", jugar(lambda obs: env.action_space.sample()))

Aleatoria: 23.36


Una política "a mano" muy simple: empujar hacia donde se está cayendo el palo.
Esto ya es una **PFA** (*Policy Function Approximation*) de Powell: una regla paramétrica
directa estado → acción, sin aprender ningún valor.

In [5]:
print("Solo ángulo:", jugar(lambda obs: int(obs[2] > 0)))
print("Ángulo + velocidad angular:", jugar(lambda obs: int(obs[2] + 0.5 * obs[3] > 0)))

Solo ángulo: 41.54
Ángulo + velocidad angular: 500.0


**Dato curioso**: solo con mirar el ángulo el palo dura ~40 pasos, pero si además miramos la
velocidad angular (hacia dónde *se está moviendo* el palo), una regla de una sola línea llega
al máximo de 500 en todos los episodios. Para CartPole, una PFA bien elegida es suficiente.
Esto ilustra un punto central de Powell: **no existe una clase de política "mejor" en general**.
Depende del problema. DQN (notebook 2) aprende algo equivalente sin que nadie le diga qué mirar,
y esa es la ventaja cuando no tenemos intuición sobre el problema (por ejemplo, con píxeles de Atari).

## Para llevar
- Gymnasium da una **interfaz estándar**: `reset()`, `step(a)`, espacios de obs/acción.
  El mismo código de agente sirve para cualquier entorno.
- Una política es solo una función `obs → acción`. Aquí ya vimos dos tipos: aleatoria y
  una regla paramétrica (PFA).
- Lo que sigue: **aprender** la política a partir de la experiencia, sin que nadie nos diga
  la regla. Notebook 1 → Blackjack con Q-learning.